In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SUBJECTS = ["Maths", "Physics", "Chemistry", "English", "Computer_Science"]
FEATURES = ["average_marks", "attendance_pct", "marks_consistency"]
RANDOM_STATE = 42


def load_data(filename):
    df = pd.read_csv(filename, dtype={"class": str})
    df["average_marks"] = df[SUBJECTS].mean(axis=1)
    df["marks_consistency"] = df[SUBJECTS].std(axis=1)
    return df


def find_best_k(X_scaled):
    k_values = [2, 3, 4, 5]
    scores = []

    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = model.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        scores.append(score)

    best_score = max(scores)
    tolerance = 0.015

    best_k = k_values[0]
    for i in range(len(k_values)):
        if scores[i] >= best_score - tolerance:
            best_k = k_values[i]
            break

    result = {
        "k_values": k_values,
        "scores": scores,
        "best_k": best_k
    }

    return result


def run_clustering(df, k, X_scaled):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    df["segment"] = model.fit_predict(X_scaled)
    return df


def label_segment(avg, attendance, consistency):
    if avg >= 70 and attendance >= 80:
        return "High Achievers"

    if avg >= 70 and attendance < 80:
        return "Talented but Disengaged"

    if avg < 50 and attendance < 70:
        return "High Risk - Struggling and Absent"

    if consistency >= 15 and avg >= 55 and attendance < 75:
        return "Uneven Performers - Attendance Gap"

    if consistency >= 15 and avg >= 55:
        return "Uneven Performers - Engaged"

    if consistency >= 15:
        return "Uneven Performers - Below Average"

    return "Steady Mid-Performers"


def build_segment_profiles(df):
    profiles = df.groupby("segment")[FEATURES].mean().round(2)
    profiles["student_count"] = df.groupby("segment").size()
    profiles = profiles.reset_index()

    profile_names = []
    for i in range(len(profiles)):
        row = profiles.iloc[i]
        name = label_segment(row["average_marks"], row["attendance_pct"], row["marks_consistency"])
        profile_names.append(name)

    profiles["profile_name"] = profile_names

    return profiles


def get_recommendation(profile_name):
    recommendations = {
        "High Achievers": "Offer enrichment content and peer-mentoring opportunities.",
        "Talented but Disengaged": "Investigate attendance barriers. Ability is already there.",
        "High Risk - Struggling and Absent": "Priority group. Needs academic support plus attendance outreach.",
        "Uneven Performers - Attendance Gap": "Prioritize attendance outreach alongside subject tutoring.",
        "Uneven Performers - Engaged": "Attendance is fine. Needs subject-specific tutoring.",
        "Uneven Performers - Below Average": "Needs broad academic support plus subject-specific tutoring.",
        "Steady Mid-Performers": "Standard support is enough. Monitor for changes over time."
    }

    if profile_name in recommendations:
        return recommendations[profile_name]

    return "Monitor this group."


def plot_results(k_search, df, filename):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(k_search["k_values"], k_search["scores"], marker="o")
    axes[0].axvline(k_search["best_k"], color="red", linestyle="--")
    axes[0].set_title("Silhouette Score by Number of Clusters")
    axes[0].set_xlabel("Number of Clusters (k)")
    axes[0].set_ylabel("Silhouette Score")

    scatter = axes[1].scatter(df["attendance_pct"], df["average_marks"], c=df["segment"], cmap="viridis")
    axes[1].set_title("Student Segments")
    axes[1].set_xlabel("Attendance %")
    axes[1].set_ylabel("Average Marks")
    plt.colorbar(scatter, ax=axes[1], label="Segment")

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


def write_report(k_search, profiles, filename):

    file = open(filename, "w")

    file.write("STUDENT SEGMENTATION REPORT\n")
    file.write("=" * 40 + "\n\n")

    file.write("Number of segments chosen: " + str(k_search["best_k"]) + "\n")
    file.write("Features used: " + ", ".join(FEATURES) + "\n\n")

    file.write("SEGMENT PROFILES\n")
    file.write("-" * 40 + "\n")

    for i in range(len(profiles)):
        row = profiles.iloc[i]

        file.write("\nSegment " + str(int(row["segment"])) + ": " + row["profile_name"])
        file.write(" (" + str(int(row["student_count"])) + " students)\n")
        file.write("  Avg marks: " + str(row["average_marks"]) + "%\n")
        file.write("  Avg attendance: " + str(row["attendance_pct"]) + "%\n")
        file.write("  Marks consistency: " + str(row["marks_consistency"]) + "\n")

        recommendation = get_recommendation(row["profile_name"])
        file.write("  Recommendation: " + recommendation + "\n")

    file.write("\n\nOVERALL TAKEAWAY\n")
    file.write("-" * 40 + "\n")
    file.write("Different segments need different support. One general plan for\n")
    file.write("everyone wastes effort. Matching the right support to the right\n")
    file.write("segment should work better for the same budget.\n")

    file.close()


def main():

    df = load_data("clean_student_marks.csv")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[FEATURES])

    k_search = find_best_k(X_scaled)
    df = run_clustering(df, k_search["best_k"], X_scaled)

    profiles = build_segment_profiles(df)

    plot_results(k_search, df, "segmentation_chart.png")
    write_report(k_search, profiles, "segmentation_report.txt")

    output_columns = ["student_id", "name", "class", "average_marks",
                       "attendance_pct", "marks_consistency", "segment"]
    df[output_columns].to_csv("segment_profiles.csv", index=False)

    print("Segmentation done!")
    print("Best k:", k_search["best_k"])
    for i in range(len(profiles)):
        row = profiles.iloc[i]
        print("Segment", int(row["segment"]), "-", row["profile_name"], "-", int(row["student_count"]), "students")
    print("Files saved: segmentation_chart.png, segment_profiles.csv, segmentation_report.txt")


if __name__ == "__main__":
    main()

C:\Users\souma\anaconda5\Lib\anaconda6\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\souma\anaconda5\Lib\anaconda6\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\souma\anaconda5\Lib\anaconda6\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\souma\anaconda5\Lib\anaconda6\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: User

Segmentation done!
Best k: 3
Segment 0 - Uneven Performers - Attendance Gap - 46 students
Segment 1 - Uneven Performers - Engaged - 48 students
Segment 2 - Steady Mid-Performers - 23 students
Files saved: segmentation_chart.png, segment_profiles.csv, segmentation_report.txt
